# 14.7 - Multi-Tool Agent

Status: VERIFIED

## What Are We Solving?
Real tasks require different capabilities: searching, calculating, writing, reading files. The agent must route to the correct tool based on context.

In [1]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # loads from .env in project root
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "qwen/qwen3.8-27b"

# Quick test
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'groq ok' only"}],
    max_tokens=10
)
print(f"Groq connected: {r.choices[0].message.content.strip()}")
print(f"Model: {MODEL}")

Groq connected: groq ok
Model: qwen/qwen3.8-27b


## Multiple Tools

In [2]:
def web_search(query: str) -> str:
    return json.dumps({"results": [f"Result 1 for '{query}'", f"Result 2 for '{query}'"]})

def calculator(expression: str) -> str:
    allowed = set("0123456789+-*/.() ")
    if not all(c in allowed for c in expression):
        return json.dumps({"error": "Invalid characters"})
    return json.dumps({"result": eval(expression)})

def read_database(query: str) -> str:
    return json.dumps({"records": [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]})

tools = [
    {"type": "function", "function": {"name": "web_search", "description": "Search the internet for current information", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "calculator", "description": "Evaluate a mathematical expression", "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
    {"type": "function", "function": {"name": "read_database", "description": "Query an internal database for stored records", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
]

tool_map = {"web_search": web_search, "calculator": calculator, "read_database": read_database}
print("Multi-tool agent ready:", list(tool_map.keys()))

Multi-tool agent ready: ['web_search', 'calculator', 'read_database']


## Multi-Tool Agent Loop

In [3]:
def run_multi_tool_agent(task: str, max_steps: int = 8) -> dict:
    messages = [
        {"role": "system", "content": (
            "You have access to web search, a calculator, and a database. "
            "Use the most appropriate tool for each part of the task. "
            "When you have enough information, provide a final answer."
        )},
        {"role": "user", "content": task}
    ]
    
    trace = []
    
    for step in range(max_steps):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools, tool_choice="auto"
        )
        msg = response.choices[0].message
        
        if msg.tool_calls:
            messages.append(msg)
            for tc in msg.tool_calls:
                name = tc.function.name
                args = json.loads(tc.function.arguments)
                result = tool_map[name](**args)
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                trace.append({"step": step + 1, "tool": name, "args": args})
                print(f"  Step {step + 1}: {name}({args})")
        else:
            trace.append({"step": step + 1, "final": True})
            return {"answer": msg.content, "steps": step + 1, "trace": trace}
    
    return {"answer": "Max steps reached.", "steps": max_steps, "trace": trace}

result = run_multi_tool_agent("What is 15 * 7 + 3?")
print(f"\nAnswer: {result['answer'][:200]}")

  Step 1: calculator({'expression': '15 * 7 + 3'})



Answer: 15 × 7 + 3 = **108**


In [4]:
# Verification
assert result["steps"] > 0
print("VERIFICATION PASSED: Phase 14.7 complete")

VERIFICATION PASSED: Phase 14.7 complete
